# 5. Serving and latency

The model scored one transaction at a time by a service that keeps a time-sorted neighbour index. Numbers come from one machine replaying historical data; they are not a production claim, and the 100 ms budget is an illustrative assumption.

In [ ]:
%matplotlib inline
from pipeline.notebook_support import *
import pandas as pd
import numpy as np
pd.set_option('display.width', 160)
import json
lat = load_results('latency.csv')
meta = json.loads((out_dir() / 'latency_meta.json').read_text())
print('FIXTURE RESULTS - not findings' if meta['fixture'] else 'real data')
print('hardware:', meta['hardware'])
print('parity with the offline scores:', meta['parity'])

In [ ]:
cols = ['model', 'device', 'mode', 'batch_size', 'concurrency', 'p50_ms', 'p95_ms', 'p99_ms', 'throughput_rps', 'p99_within_budget']
lat[cols].round(2)

In [ ]:
plot_latency(lat, meta['budget_ms']);

## Where the time goes (batch size 1)

In [ ]:
plot_breakdown(lat);

## Stock sampler versus the custom index

The `stock_neighbor_loader` row is the cost of sampling the neighbourhood alone with PyG's loader built per request; compare it with the `neighbour_lookup` share above.

In [ ]:
lat[lat['mode'] == 'lookup_only'][['model', 'p50_ms', 'p95_ms', 'p99_ms']].round(2)

Index size and the concurrency caveat: the metadata printed above lists the indexed transactions and the serialised index size. Scoring is serialised by one lock, so extra clients do not add throughput; the multi-client row shows the queueing cost.